### 1. Dependencies

In [8]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### 2. Imports, config & model

In [2]:
import os, fitz, base64, json, re
from pathlib import Path
from io import BytesIO
from PIL import Image
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextBox
from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download
from portkey_ai import Portkey

PDF_PATH   = Path("..\\..\\Sample PDFs\\InfoEdge_Annual_Report_2025.pdf")
OUTPUT_DIR = Path("Output")
OUTPUT_DIR.mkdir(exist_ok=True)

client = Portkey(
    base_url = os.environ.get("PORTKEY_BASE_URL"),
    api_key=os.environ.get("PORTKEY_API_KEY")
)

model_path = hf_hub_download(
    repo_id  = "juliozhao/DocLayout-YOLO-DocStructBench",
    filename = "doclayout_yolo_docstructbench_imgsz1024.pt",
)
model = YOLOv10(model_path)

print("Ready:", PDF_PATH)

C:\Users\78235\AppData\Roaming\Python\Python312\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\78235\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready: ..\..\Sample PDFs\InfoEdge_Annual_Report_2025.pdf


### 3. Prompts

In [3]:
SYSTEM_PROMPT = (
    "You are a document analysis assistant. "
    "Classify and describe visual elements from documents with precision. "
    "Always respond in the exact structured format requested — no extra text."
)

USER_PROMPT = """\
Caption from document: '{caption}'

Step 1 — Classify: is this a TABLE (rows/columns) or a FIGURE (chart, graph, diagram)?
Step 2 — Extract the heading exactly as written (e.g. 'Figure 3', 'Exhibit 1').
Step 3 — Count distinct sub-plots. If multiple, label them (a), (b), (c)…
Step 4 — Reply in EXACTLY this format, nothing else:

KEY: <TABLE or FIGURE>_<number>
HEADING: <exact caption heading>
CONTENT:
<For TABLE: faithful markdown table.>
<For FIGURE single plot: 2-5 sentence description.>
<For FIGURE multi-plot: one overall sentence, then **(a)** … **(b)** …>
"""

print("Prompts set.")

Prompts set.


### 4. Detect visual boxes and merge figure + caption pairs

In [4]:
VISUAL_CLASSES  = {3: "Figure", 4: "Figure", 5: "Table", 6: "Table"}
CAPTION_CLASSES = {4, 6}

def detect_and_merge(page_img):
    W, H = page_img.size
    raw = []
    for r in model.predict(page_img, imgsz=1024, conf=0.25, device="cpu"):
        for box in r.boxes:
            cls = int(box.cls)
            if cls not in VISUAL_CLASSES:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            raw.append({"cls": cls, "label": VISUAL_CLASSES[cls],
                        "box": (x1/W, y1/H, x2/W, y2/H)})

    figures  = [d for d in raw if d["cls"] in {3, 5}]
    captions = [d for d in raw if d["cls"] in CAPTION_CLASSES]
    merged, used = [], set()

    for fig in figures:
        fx1, fy1, fx2, fy2 = fig["box"]
        best, best_dist = None, float("inf")
        for j, cap in enumerate(captions):
            cx1, cy1, cx2, cy2 = cap["box"]
            h_overlap = min(fx2, cx2) - max(fx1, cx1)
            v_dist    = max(fy1 - cy2, cy1 - fy2, 0)
            if h_overlap > 0 and v_dist < 0.15 and v_dist < best_dist:
                best_dist, best = v_dist, (j, cap)
        if best:
            j, cap = best
            used.add(j)
            cx1, cy1, cx2, cy2 = cap["box"]
            merged_box = (min(fx1,cx1), min(fy1,cy1), max(fx2,cx2), max(fy2,cy2))
            merged.append({**fig, "box": merged_box})
        else:
            merged.append(fig)

    for j, cap in enumerate(captions):
        if j not in used:
            merged.append(cap)

    return merged

print("detect_and_merge ready.")

detect_and_merge ready.


### 5. Extract text blocks from a page

In [5]:
def get_text_blocks(pdf_path, page_index):
    blocks = []
    for layout in extract_pages(str(pdf_path), page_numbers=[page_index]):
        W, H = layout.width, layout.height
        for el in layout:
            if isinstance(el, LTTextBox):
                text = el.get_text().strip()
                if not text:
                    continue
                x0, y0, x1, y1 = el.bbox
                blocks.append({"text": text,
                                "box":  (x0/W, 1 - y1/H, x1/W, 1 - y0/H)})
    return blocks

def overlap(tbox, dbox):
    ax1, ay1, ax2, ay2 = tbox
    bx1, by1, bx2, by2 = dbox
    inter = max(0, min(ax2,bx2) - max(ax1,bx1)) * max(0, min(ay2,by2) - max(ay1,by1))
    area  = (ax2 - ax1) * (ay2 - ay1)
    return inter / area if area > 0 else 0.0

print("get_text_blocks + overlap ready.")

get_text_blocks + overlap ready.


### 6. Get caption text for a detection

In [6]:
def get_caption(det, text_blocks, all_dets):
    det_box     = det["box"]
    overlapping = [b for b in text_blocks if overlap(b["box"], det_box) > 0.4]

    if det["cls"] in CAPTION_CLASSES:
        return " ".join(b["text"] for b in overlapping)

    cx = (det_box[0] + det_box[2]) / 2
    for d2 in all_dets:
        if d2["cls"] not in CAPTION_CLASSES:
            continue
        if abs(cx - (d2["box"][0] + d2["box"][2]) / 2) < 0.3:
            cap_blocks = [b for b in text_blocks if overlap(b["box"], d2["box"]) > 0.4]
            if cap_blocks:
                return cap_blocks[0]["text"]
    return ""

print("get_caption ready.")

get_caption ready.


### 7. Call Vision Model on a cropped image

In [7]:
def ask_vision(page_img, det, caption_text, counter):
    W, H = page_img.size
    x1, y1, x2, y2 = det["box"]
    crop = page_img.crop((int(x1*W), int(y1*H), int(x2*W), int(y2*H)))

    buf = BytesIO()
    crop.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()

    resp = client.chat.completions.create(
        model="gpt-4o",
        max_tokens=1200,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text",      "text": USER_PROMPT.format(caption=caption_text)},
            ]},
        ],
    )
    raw = resp.choices[0].message.content.strip()

    key_m     = re.search(r"KEY:\s*(.+)",          raw)
    heading_m = re.search(r"HEADING:\s*(.+)",      raw)
    content_m = re.search(r"CONTENT:\s*([\s\S]+)", raw)
    label     = det["label"]

    key     = key_m.group(1).strip()     if key_m     else f"{label.upper()}_{counter}"
    heading = heading_m.group(1).strip() if heading_m else ""
    content = content_m.group(1).strip() if content_m else raw

    sub_plots = {
        m.group(1): m.group(2).strip()
        for m in re.finditer(r"\*\*\(([a-z])\)\*\*\s*([\s\S]*?)(?=\*\*\([a-z]\)\*\*|$)", content)
    }

    return {
        "key": key, "heading": heading, "content": content,
        "caption": caption_text, "sub_plots": sub_plots or None,
    }

print("ask_vision ready.")

ask_vision ready.


### 8. Enrich markdown — load JSON and replace Figure / Table references

In [8]:
try:
    import regex as _re
except ImportError:
    import re as _re


def replace(m , lookup):
    key = m.group(1)
    return f"{key} ({lookup[key]})" if key in lookup else key

def enrich_markdown(md_path, json_path, out_path):
    figures = json.loads(Path(json_path).read_text(encoding="utf-8"))

    lookup = {}
    for fig in figures:
        kind = fig.get("key", "").split("_")[0].capitalize()
        num  = str(fig.get("number") or "").strip()
        if not num:
            continue
        if fig.get("content"):
            lookup[f"{kind} {num}"] = fig["content"]
        for sub, sub_content in (fig.get("sub_plots") or {}).items():
            lookup[f"{kind} {num}{sub}"] = sub_content

    md_text = Path(md_path).read_text(encoding="utf-8")

    pattern = r"\b((?:Figure|Table)\s+\d+[a-zA-Z]?)\b(?!\s*:)"
    

    enriched = _re.sub(pattern, lambda m: replace(m, lookup), md_text)
    Path(out_path).write_text(enriched, encoding="utf-8")
    print(f"Enriched → {out_path}  ({len(lookup)} entries)")

print("enrich_markdown ready.")

enrich_markdown ready.


In [9]:
import json
from pathlib import Path

try:
    import regex as _re
except ImportError:
    import re as _re


def enrich_markdown(md_path, json_path, out_path):
    figures = json.loads(Path(json_path).read_text(encoding="utf-8"))

    # Build dictionary in your desired format
    lookup = {}
    for item in figures:
        main_key = item.get("key")
        if main_key and item.get("content"):
            lookup[main_key] = item["content"]

        if item.get("sub_plots"):
            for sub_key, sub_content in item["sub_plots"].items():
                lookup[f"{main_key}_{sub_key}"] = sub_content

    md_text = Path(md_path).read_text(encoding="utf-8")

    # Matches: Figure 1, Figure 1a, Table 2, Table 2b
    pattern = r"\b((?:Figure|Table)\s+\d+[a-zA-Z]?)\b(?!\s*:)"

    def ref_to_lookup_key(ref):
        # Convert:
        # "Figure 1"  -> "FIGURE_1"
        # "Figure 1a" -> "FIGURE_1_a"
        # "Table 2"   -> "TABLE_2"
        # "Table 2b"  -> "TABLE_2_b"
        m = _re.match(r"(Figure|Table)\s+(\d+)([a-zA-Z]?)", ref, flags=_re.IGNORECASE)
        if not m:
            return None

        kind = m.group(1).upper()
        number = m.group(2)
        subplot = m.group(3).lower()

        if subplot:
            return f"{kind}_{number}_{subplot}"
        return f"{kind}_{number}"

    def replace(m):
        ref = m.group(1)
        lookup_key = ref_to_lookup_key(ref)
        return f"{ref} ({lookup[lookup_key]})" if lookup_key in lookup else ref

    enriched = _re.sub(pattern, replace, md_text)
    Path(out_path).write_text(enriched, encoding="utf-8")
    print(f"Enriched -> {out_path} ({len(lookup)} entries)")


print("enrich_markdown ready.")

enrich_markdown ready.


### 9. Run pipeline

In [10]:
md_path   = OUTPUT_DIR / f"{PDF_PATH.stem}.md"
json_path = OUTPUT_DIR / f"{PDF_PATH.stem}.json"
enr_path  = OUTPUT_DIR / f"{PDF_PATH.stem}_enriched.md"

doc = fitz.open(str(PDF_PATH))
all_pages_md, figure_refs, counter = [], [], 0

for page_idx in range(len(doc)):
    print(f"\nPage {page_idx + 1}/{len(doc)}")

    page = doc[page_idx]
    pix  = page.get_pixmap(matrix=fitz.Matrix(150/72, 150/72))
    img  = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    detections  = detect_and_merge(img)
    text_blocks = get_text_blocks(PDF_PATH, page_idx)
    print(f"  {len(detections)} detections, {len(text_blocks)} text blocks")

    removed = set()
    for det in detections:
        hits = [(i, b) for i, b in enumerate(text_blocks)
                if overlap(b["box"], det["box"]) > 0.4]
        if not hits:
            continue
        counter += 1
        caption_text = get_caption(det, text_blocks, detections)
        try:
            meta = ask_vision(img, det, caption_text, counter)
            meta["page"]   = page_idx + 1
            meta["number"] = counter
            figure_refs.append(meta)
            replacement = f"{meta['key']} : {meta['content']}"
            print(f"  {meta['key']} ✓")
        except Exception as e:
            replacement = f"({det['label']} {counter} — vision error: {e})"
            print(f"  Error: {e}")
        first_idx = hits[0][0]
        for i, _ in hits:
            removed.add(i)
        removed.discard(first_idx)
        text_blocks[first_idx]["replacement"] = replacement

    lines = [b.get("replacement", b["text"])
             for i, b in enumerate(text_blocks) if i not in removed]
    all_pages_md.append(f"## Page {page_idx + 1}\n\n" + "\n\n".join(lines))

doc.close()

md_path.write_text("\n\n---\n\n".join(all_pages_md), encoding="utf-8")
json_path.write_text(json.dumps(figure_refs, indent=2), encoding="utf-8")
print(f"\nSaved: {md_path}  |  {json_path}")

enrich_markdown(md_path, json_path, enr_path)
print(f"Done! {len(figure_refs)} visuals processed.")


Page 1/207

0: 1024x736 2 titles, 1 plain text, 2 abandons, 1 figure, 2996.0ms
Speed: 11.7ms preprocess, 2996.0ms inference, 0.0ms postprocess per image at shape (1, 3, 1024, 736)
  1 detections, 0 text blocks

Page 2/207

0: 736x1024 3 titles, 18 plain texts, 3136.7ms
Speed: 23.2ms preprocess, 3136.7ms inference, 14.5ms postprocess per image at shape (1, 3, 736, 1024)
  0 detections, 69 text blocks

Page 3/207

0: 736x1024 2 titles, 10 plain texts, 4 abandons, 2146.3ms
Speed: 38.0ms preprocess, 2146.3ms inference, 0.0ms postprocess per image at shape (1, 3, 736, 1024)
  0 detections, 15 text blocks

Page 4/207

0: 736x1024 3 titles, 11 plain texts, 4 abandons, 1 figure, 2 figure_captions, 1 table_footnote, 2689.3ms
Speed: 9.5ms preprocess, 2689.3ms inference, 7.7ms postprocess per image at shape (1, 3, 736, 1024)
  2 detections, 45 text blocks
  FIGURE_1 ✓
  FIGURE_2 ✓

Page 5/207

0: 736x1024 8 titles, 9 plain texts, 8 abandons, 2 figures, 4088.3ms
Speed: 11.9ms preprocess, 4088.3ms

### 10. Run parallel pipeline *(optional)*

In [9]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS   = 4
par_md_path   = OUTPUT_DIR / f"{PDF_PATH.stem}_parallel.md"
par_json_path = OUTPUT_DIR / f"{PDF_PATH.stem}_parallel.json"
par_enr_path  = OUTPUT_DIR / f"{PDF_PATH.stem}_parallel_enriched.md"

def process_page(page_idx):
    doc2 = fitz.open(str(PDF_PATH))
    pix  = doc2[page_idx].get_pixmap(matrix=fitz.Matrix(150/72, 150/72))
    img  = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc2.close()

    detections  = detect_and_merge(img)
    text_blocks = get_text_blocks(PDF_PATH, page_idx)
    removed, page_refs, local_counter = set(), [], 0

    for det in detections:
        hits = [(i, b) for i, b in enumerate(text_blocks)
                if overlap(b["box"], det["box"]) > 0.4]
        if not hits:
            continue
        local_counter += 1
        caption_text = get_caption(det, text_blocks, detections)
        try:
            meta = ask_vision(img, det, caption_text, local_counter)
            meta["page"]   = page_idx + 1
            meta["number"] = local_counter
            page_refs.append(meta)
            replacement = f"{meta['key']} : {meta['content']}"
        except Exception as e:
            replacement = f"({det['label']} {local_counter} — error: {e})"
        first_idx = hits[0][0]
        for i, _ in hits:
            removed.add(i)
        removed.discard(first_idx)
        text_blocks[first_idx]["replacement"] = replacement

    lines   = [b.get("replacement", b["text"])
               for i, b in enumerate(text_blocks) if i not in removed]
    page_md = f"## Page {page_idx + 1}\n\n" + "\n\n".join(lines)
    return page_idx, page_md, page_refs


total_pages = len(fitz.open(str(PDF_PATH)))
results     = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_page, i): i for i in range(total_pages)}
    for future in as_completed(futures):
        idx, page_md, page_refs = future.result()
        results[idx] = (page_md, page_refs)
        print(f"Page {idx + 1} done, {len(page_refs)} visuals")

all_par_md   = [results[i][0] for i in range(total_pages)]
all_par_refs = [ref for i in range(total_pages) for ref in results[i][1]]

par_md_path.write_text("\n\n---\n\n".join(all_par_md), encoding="utf-8")
par_json_path.write_text(json.dumps(all_par_refs, indent=2), encoding="utf-8")
print(f"Saved: {par_md_path}  |  {par_json_path}")

enrich_markdown(par_md_path, par_json_path, par_enr_path)
print(f"Parallel done! {len(all_par_refs)} total visuals.")


Ultralytics YOLOv0.0.4 🚀 Python-3.12.10 torch-2.10.0+cpu CPU (11th Gen Intel Core(TM) i7-1185G7 3.00GHz)



0: 736x1024 3 titles, 18 plain texts, 2352.7ms
Speed: 15.6ms preprocess, 2352.7ms inference, 3.0ms postprocess per image at shape (1, 3, 736, 1024)
Ultralytics YOLOv0.0.4 🚀 Python-3.12.10 torch-2.10.0+cpu CPU (11th Gen Intel Core(TM) i7-1185G7 3.00GHz)
Page 2 done, 0 visuals

0: 1024x736 2 titles, 1 plain text, 2 abandons, 1 figure, 3251.1ms
Speed: 43.4ms preprocess, 3251.1ms inference, 3.1ms postprocess per image at shape (1, 3, 1024, 736)
Page 1 done, 0 visuals

0: 736x1024 3 titles, 11 plain texts, 4 abandons, 1 figure, 2 figure_captions, 1 table_footnote, 2276.9ms
Speed: 31.4ms preprocess, 2276.9ms inference, 1.4ms postprocess per image at shape (1, 3, 736, 1024)
0: 736x1024 8 titles, 9 plain texts, 8 abandons, 2 figures, 2657.5ms
Speed: 16.5ms preprocess, 2657.5ms inference, 3.0ms postprocess per image at shape (1, 3, 736, 1024)
0: 736x1024 2 titles, 10 plain texts, 4 aband